# 03 — Evaluation
Evaluate the trained model on the held-out test set.

**Requires:** `face_mask_classifier.pth` and `test_indices.pkl` from notebook 02.

In [ ]:
import kagglehub
path = kagglehub.dataset_download('vijaykumar1799/face-mask-detection')

from google.colab import files
print('Upload face_mask_classifier.pth and test_indices.pkl:')
files.upload()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pickle

DATA_DIR   = path
CLASS_NAMES = ['mask_weared_incorrect', 'with_mask', 'without_mask']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(DATA_DIR, transform=eval_transforms)

with open('test_indices.pkl', 'rb') as f:
    test_indices = pickle.load(f)

test_set  = Subset(full_dataset, test_indices)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 3)
model.load_state_dict(torch.load('face_mask_classifier.pth', map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print('Model loaded.')

## Classification Report

In [ ]:
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Save Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average='macro')
rec  = recall_score(all_labels, all_preds, average='macro')
f1   = f1_score(all_labels, all_preds, average='macro')

metrics = f"""Face Mask Detection — Test Set Metrics
========================================
Accuracy:          {acc:.4f} ({acc*100:.2f}%)
Precision (macro): {prec:.4f}
Recall (macro):    {rec:.4f}
F1 Score (macro):  {f1:.4f}
Test samples:      {len(all_labels)}
Misclassified:     {sum(p != l for p, l in zip(all_preds, all_labels))}
"""

print(metrics)
with open('metrics.txt', 'w') as f:
    f.write(metrics)
print('Saved metrics.txt')